# Kline Data Analysis and Smoothing

This notebook demonstrates how to load a kline Parquet file, apply a Savitzky-Golay filter to smooth the closing price, and visualize the results.

In [ ]:
import polars as pl
import plotly.graph_objects as go
from scipy.signal import savgol_filter
import os

# Define the path to a sample kline file.
# IMPORTANT: You will need to change this path to point to a valid Parquet file in your `data/klines` directory.
KLINE_FILE_PATH = "../../../data/klines/BTCUSDT_1m_2024-08_2025-09.parquet"

# Check if the file exists before trying to read it
if not os.path.exists(KLINE_FILE_PATH):
    raise FileNotFoundError(
        f"Error: The sample kline file was not found at '{KLINE_FILE_PATH}'. "
        "Please update the KLINE_FILE_PATH variable to point to a valid Parquet file."
    )

# Load the kline data from the Parquet file
df = pl.read_parquet(KLINE_FILE_PATH)

print("Successfully loaded the dataset. Here are the first 5 rows:")
print(df.head())

## Apply Savitzky-Golay Smoothing

Now, we'll apply a Savitzky-Golay filter to the `close` price column. This is a common technique for smoothing noisy time-series data, helping to reveal underlying trends. We will use a window length of 51 and a polynomial order of 3, which are common starting parameters.

In [ ]:
# Ensure the 'close' column is a numeric type (float)
df = df.with_columns(pl.col("close").cast(pl.Float64))

# Apply the Savitzky-Golay filter
# Parameters: window_length=51, polyorder=3
# window_length must be odd.
window_length = 51
poly_order = 3

# The savgol_filter function from scipy works on NumPy arrays
close_prices = df["close"].to_numpy()
smoothed_close_prices = savgol_filter(close_prices, window_length, poly_order)

# Add the smoothed data back to the DataFrame as a new column
df = df.with_columns(pl.Series(name="close_smoothed", values=smoothed_close_prices))

print("Added 'close_smoothed' column. Here are the first 5 rows with the new column:")
print(df.head())

## Visualize the Results

Finally, let's plot the original closing price against the smoothed closing price. This will allow us to visually inspect the effect of the filter. We'll use Plotly for an interactive visualization.

In [ ]:
# Create an interactive plot with Plotly
fig = go.Figure()

# Add the original 'close' price trace
fig.add_trace(go.Scatter(
    x=df["open_time"],
    y=df["close"],
    mode='lines',
    name='Original Close Price',
    line=dict(color='blue', width=1)
))

# Add the smoothed 'close' price trace
fig.add_trace(go.Scatter(
    x=df["open_time"],
    y=df["close_smoothed"],
    mode='lines',
    name=f'Smoothed Close (Window={window_length}, Order={poly_order})',
    line=dict(color='red', width=2)
))

# Update layout for a clean look
fig.update_layout(
    title="Original vs. Smoothed Close Price for BTCUSDT",
    xaxis_title="Time",
    yaxis_title="Price (USDT)",
    legend_title="Trace",
    template="plotly_white"
)

# Show the plot
fig.show()